# Data cleaning RR. HH. 13/07/2026

## Importación de librerías

Carga pandas, librerías gráficas, utilidades de fecha, lectura oculta de contraseñas y SQLAlchemy.

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv

import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

## Conexión y carga desde MySQL

In [2]:
SOURCE_TABLE = "RRHH_13072026"

load_dotenv(dotenv_path=".env") 

connection_url = URL.create(
    drivername="mysql+mysqlconnector",
    username=os.environ["MYSQL_USER"],
    password=os.environ["MYSQL_PASSWORD"],
    host=os.environ["MYSQL_HOST"],
    port=int(os.environ["MYSQL_PORT"]),
    database=os.environ["MYSQL_DATABASE"],
)

engine = create_engine(connection_url, pool_pre_ping=True)



df_RRHH = pd.read_sql_table(SOURCE_TABLE, con=engine)

print(f"{SOURCE_TABLE}: {df_RRHH.shape[0]} filas, "
      f"{df_RRHH.shape[1]} columnas")


RRHH_13072026: 1110 filas, 21 columnas


## Inspección visual inicial

In [3]:
df_RRHH.head()

,ID,Reason_absence,Month_absence,Day_week,Seasons,Transportation_expense,Distance_Residence_Work,Service_time,Age,Work_load_Average_day,...,Disciplinary_failure,Education,Son,Social_drinker,Social_smoker,Pet,Weight,Height,Body_mass_index,Absenteeism_hours
0,14,11,11,2,4,155,12,14,34,"284,031",...,0,1,2,1,0,0,95,196,25,120
1,36,13,4,4,3,118,13,18,50,"239,409",...,0,1,1,1,0,0,98,178,31,120
2,9,6,7,3,1,228,14,16,58,"264,604",...,0,1,2,0,0,1,65,172,22,120
3,28,9,7,3,1,225,26,9,28,"230,290",...,0,1,1,0,0,2,69,169,24,112
4,9,12,3,3,2,228,14,16,58,"222,196",...,0,1,2,0,0,1,65,172,22,112


## Eliminación de duplicados exactos

Cuenta las filas completamente duplicadas, las elimina y reinicia el índice. Se registra las filas eliminadas y se verifica que el resultado ya no contiene duplicados.

In [4]:
# Identificar y conservar los duplicados antes de eliminarlos.
mask_duplicated = df_RRHH.duplicated(keep="first")
duplicados_eliminados = df_RRHH.loc[mask_duplicated].copy()

df_RRHH = (
    df_RRHH.loc[~mask_duplicated]
    .reset_index(drop=True)
    .copy()
)

assert not df_RRHH.duplicated().any()
print(f"Duplicados exactos eliminados: {len(duplicados_eliminados)}")
print(f"Filas después de deduplicar: {len(df_RRHH)}")


Duplicados exactos eliminados: 54
Filas después de deduplicar: 1056


## Conversión de tipos

Convierte con diagnóstico explícito y evita que valores no numéricos pasen inadvertidos.

In [5]:
df_RRHH.columns


Index(['ID', 'Reason_absence', 'Month_absence', 'Day_week', 'Seasons',
       'Transportation_expense', 'Distance_Residence_Work', 'Service_time',
       'Age', 'Work_load_Average_day', 'Hit_target', 'Disciplinary_failure',
       'Education', 'Son', 'Social_drinker', 'Social_smoker', 'Pet', 'Weight',
       'Height', 'Body_mass_index', 'Absenteeism_hours'],
      dtype='str')

In [6]:
#Guardamos el original para poder detectar valores problematicos en el futuro.
df_original = df_RRHH.copy()

# Normalizar la coma decimal antes de aplicar la transformación.
df_RRHH["Work_load_Average_day"] = (
    df_RRHH["Work_load_Average_day"]
    .astype("string")
    .str.replace(",", ".", regex=False)
)

float_columns = ["Transportation_expense", "Work_load_Average_day"]
int_columns = [
    "ID", "Reason_absence", "Month_absence",
    "Day_week", "Seasons", "Distance_Residence_Work", "Service_time",
    "Age", "Hit_target", "Education", "Son","Pet", "Weight", "Height",
    "Body_mass_index", "Absenteeism_hours",
]
bool_columns = ["Disciplinary_failure", "Social_drinker", "Social_smoker"]

df_RRHH[float_columns] = df_RRHH[float_columns].apply(
    pd.to_numeric,
    errors="coerce"
).astype("Float64")

df_RRHH[int_columns] = df_RRHH[int_columns].apply(
    pd.to_numeric,
    errors="coerce"
).astype("Int64")

df_RRHH[bool_columns] = (
    df_RRHH[bool_columns]
    .replace({"1": True, "0": False,})
    .astype("boolean")
)


## Validación de conversión

In [7]:
columns_to_check = float_columns + int_columns + bool_columns

# Valores que existían originalmente, pero quedaron como nulos
invalid_mask = (
    df_original[columns_to_check].notna()
    & df_RRHH[columns_to_check].isna()
)

# Número de valores no válidos por columna
if invalid_mask.sum().sum() == 0:
    print("No se han detectado valores no válidos.")
else:
    invalid_records = []
    for column in columns_to_check:
        mask = invalid_mask[column]

        if mask.any():
            invalid_records.append(
                pd.DataFrame({
                    "row_index": df_RRHH.index[mask],
                    "column": column,
                    "original_value": df_original.loc[mask, column].values,
                })
            )

    invalid_report = pd.concat(invalid_records, ignore_index=True)
    display(invalid_report)


No se han detectado valores no válidos.


## Diccionarios y etiquetas descriptivas


In [8]:
day_week_map = {
    2: "Lunes",
    3: "Martes",
    4: "Miércoles",
    5: "Jueves",
    6: "Viernes"}

education_map = {
    1: "Educación secundaria",
    2: "Graduado",
    3: "Postgrado",
    4: "Máster o doctorado"}

month_to_season_code = {
    0: 0,
    1: 1, 2: 1, 12: 1,
    3: 2, 4: 2, 5: 2,
    6: 3, 7: 3, 8: 3,
    9: 4, 10: 4, 11: 4}

seasons_map = {
    0: "Sin mes registrado",
    1: "Invierno",
    2: "Primavera",
    3: "Verano",
    4: "Otoño"}

reason_absence_map = {
    0: "Sin ausencia registrada",
    1: "Enfermedades infecciosas y parasitarias",
    2: "Neoplasias (tumores)",
    3: "Enfermedades de la sangre y del sistema inmunitario",
    4: "Enfermedades endocrinas, nutricionales y metabólicas",
    5: "Trastornos mentales y del comportamiento",
    6: "Enfermedades del sistema nervioso",
    7: "Enfermedades del ojo",
    8: "Enfermedades del oído",
    9: "Enfermedades del sistema circulatorio",
    10: "Enfermedades del sistema respiratorio",
    11: "Enfermedades del sistema digestivo",
    12: "Enfermedades de la piel",
    13: "Enfermedades del sistema musculoesquelético",
    14: "Enfermedades del sistema genito-urinario",
    15: "Embarazo, parto y puerperio",
    16: "Afecciones del periodo perinatal",
    17: "Malformaciones congénitas",
    18: "Síntomas y hallazgos clínicos no clasificados",
    19: "Lesiones y envenenamientos",
    20: "Causas externas de morbilidad",
    21: "Factores que influyen en el estado de salud",
    22: "Seguimiento de paciente",
    23: "Consulta médica",
    24: "Donación de sangre",
    25: "Examen de laboratorio",
    26: "Ausencia injustificada",
    27: "Fisioterapia",
    28: "Consulta dental"}

In [9]:
# Aplicar cada mapa y detener el proceso si aparece un código desconocido.
mapping_specs = {
    "Day_week": ("Day_week_desc", day_week_map),
    "Education": ("Education_desc", education_map),
    "Reason_absence": ("Reason_absence_desc", reason_absence_map),
    # Mapeo de meses a estaciones de España (Originalmente estaciones de Brasil)
    "Month_absence": ("Seasons", month_to_season_code),
    "Seasons": ("Season_desc", seasons_map)}


for code_column, (description_column, mapping) in mapping_specs.items():
    
    mapped = df_RRHH[code_column].map(mapping)
    if mapped.isna().any():
        unknown = sorted(
            df_RRHH.loc[mapped.isna(), code_column].unique().tolist()
        )
        raise ValueError(f"Códigos sin mapear en {code_column}: {unknown}")
    df_RRHH[description_column] = mapped


## Inspección visual del mapeo

In [10]:
display( df_RRHH.groupby(["Seasons", "Season_desc", "Month_absence"],as_index=False).size().sort_values(["Month_absence"]))
df_RRHH.head(10)

,Seasons,Season_desc,Month_absence,size
0,0,Sin mes registrado,0,4
1,1,Invierno,1,65
2,1,Invierno,2,95
4,2,Primavera,3,122
5,2,Primavera,4,77
6,2,Primavera,5,92
7,3,Verano,6,84
8,3,Verano,7,97
9,3,Verano,8,83
10,4,Otoño,9,71


,ID,Reason_absence,Month_absence,Day_week,Seasons,Transportation_expense,Distance_Residence_Work,Service_time,Age,Work_load_Average_day,...,Social_smoker,Pet,Weight,Height,Body_mass_index,Absenteeism_hours,Day_week_desc,Education_desc,Reason_absence_desc,Season_desc
0,14,11,11,2,4,155.0,12,14,34,284.031,...,False,0,95,196,25,120,Lunes,Educación secundaria,Enfermedades del sistema digestivo,Otoño
1,36,13,4,4,2,118.0,13,18,50,239.409,...,False,0,98,178,31,120,Miércoles,Educación secundaria,Enfermedades del sistema musculoesquelético,Primavera
2,9,6,7,3,3,228.0,14,16,58,264.604,...,False,1,65,172,22,120,Martes,Educación secundaria,Enfermedades del sistema nervioso,Verano
3,28,9,7,3,3,225.0,26,9,28,230.29,...,False,2,69,169,24,112,Martes,Educación secundaria,Enfermedades del sistema circulatorio,Verano
4,9,12,3,3,2,228.0,14,16,58,222.196,...,False,1,65,172,22,112,Martes,Educación secundaria,Enfermedades de la piel,Primavera
5,11,19,3,2,2,289.0,36,13,33,222.196,...,False,1,90,172,30,104,Lunes,Educación secundaria,Lesiones y envenenamientos,Primavera
6,36,13,6,4,3,118.0,13,18,50,377.55,...,False,0,98,178,31,80,Miércoles,Educación secundaria,Enfermedades del sistema musculoesquelético,Verano
7,14,18,12,3,1,155.0,12,14,34,280.549,...,False,0,95,196,25,80,Martes,Educación secundaria,Síntomas y hallazgos clínicos no clasificados,Invierno
8,13,13,7,2,3,369.0,17,12,31,264.604,...,False,0,70,169,25,80,Lunes,Educación secundaria,Enfermedades del sistema musculoesquelético,Verano
9,34,19,12,3,1,118.0,10,10,37,261.306,...,False,0,83,172,28,64,Martes,Educación secundaria,Lesiones y envenenamientos,Invierno


## Reordenación de columnas

Coloca cada descripción junto a su código y conserva el resto de columnas en su orden relativo.


In [11]:
# Reordenar columnas para poner cada columna descriptiva al lado de su código
pares = {
    "Day_week": "Day_week_desc",
    "Education": "Education_desc",
    "Seasons": "Season_desc",
    "Reason_absence": "Reason_absence_desc"
}

nuevo_orden = []

for col in df_RRHH.columns:
    if col in pares:
        nuevo_orden.append(col)
        nuevo_orden.append(pares[col])
    elif col not in pares.values():
        nuevo_orden.append(col)

df_RRHH = df_RRHH[nuevo_orden]


In [12]:
df_RRHH.columns

Index(['ID', 'Reason_absence', 'Reason_absence_desc', 'Month_absence',
       'Day_week', 'Day_week_desc', 'Seasons', 'Season_desc',
       'Transportation_expense', 'Distance_Residence_Work', 'Service_time',
       'Age', 'Work_load_Average_day', 'Hit_target', 'Disciplinary_failure',
       'Education', 'Education_desc', 'Son', 'Social_drinker', 'Social_smoker',
       'Pet', 'Weight', 'Height', 'Body_mass_index', 'Absenteeism_hours'],
      dtype='str')

## Normalización de nombres de columnas y creación de índice único de registro.

Normaliza primero los nombres, crea un identificador único de forma controlada.

In [13]:

rename_columns = {
    "ID": "ID_Employee",
    "Reason_absence": "Reason_Absence",
    "Reason_absence_desc": "Reason_Absence_Desc",
    "Month_absence": "Month_Absence",
    "Day_week": "Day_Week",
    "Day_week_desc": "Day_Week_Desc",
    "Season_desc": "Season_Desc",
    "Transportation_expense": "Transportation_Expense",
    "Service_time": "Service_Time",
    "Work_load_Average_day": "Work_load_Average_Day",
    "Hit_target": "Hit_Target",
    "Disciplinary_failure": "Disciplinary_Failure",
    "Education_desc": "Education_Desc",
    "Social_drinker": "Social_Drinker",
    "Social_smoker": "Social_Smoker",
    "Body_mass_index": "Body_Mass_Index",
    "Absenteeism_hours": "Absenteeism_Hours",
}

df_RRHH = df_RRHH.rename(columns=rename_columns)

df_RRHH.insert(0, "ID_Absence", range(len(df_RRHH)))
assert df_RRHH["ID_Absence"].is_unique


## Validación de coherencia de los registros de empleados

Esta validación compara los atributos personales de  los empleados.

El proceso se divide en dos pasos:

1. **Detección y revisión:** identifica empleados con perfiles personales diferentes y muestra todas sus filas para poder compararlas.
2. **Confirmación y eliminación:** propone como candidatos únicamente los perfiles minoritarios cuando existe un perfil claramente mayoritario. La eliminación permanece desactivada hasta cambiar expresamente **CONFIRMAR_ELIMINACION** a **True**.

Si un empleado tiene perfiles empatados o no existe evidencia suficiente para decidir cuál es correcto, se clasifica como **caso ambiguo** y no se propone su eliminación automática.


In [14]:
# Columnas que deberían describir de forma coherente al mismo empleado.

personal_columns = [
    "Transportation_Expense", "Distance_Residence_Work", "Service_Time",
    "Age", "Education", "Son", "Social_Drinker", "Social_Smoker",
    "Pet", "Weight", "Height", "Body_Mass_Index",
]

review_frames = []
candidate_indices = []
ambiguous_employee_ids = []
inconsistent_fields_by_employee = {}

for employee_id, employee_rows in df_RRHH.groupby("ID_Employee", dropna=False):
    # Columnas que toman más de un valor para este empleado. dropna=False
    # considera también una combinación valor/nulo como posible incoherencia.
    varying_fields = [
        column
        for column in personal_columns
        if employee_rows[column].nunique(dropna=False) > 1
    ]
    if not varying_fields:
        continue

    inconsistent_fields_by_employee[employee_id] = varying_fields

    # Un hash representa la combinación completa de atributos personales de
    # cada fila y permite contar cuántas veces se repite cada perfil.
    profile_hash = pd.util.hash_pandas_object(
        employee_rows[personal_columns],
        index=False,
    )
    profile_counts = profile_hash.value_counts()

    # Solo se propone eliminar cuando hay al menos tres registros, el perfil
    # principal aparece dos o más veces y supera estrictamente al segundo.
    has_clear_majority = (
        len(employee_rows) >= 3
        and profile_counts.iloc[0] >= 2
        and (len(profile_counts) == 1 or profile_counts.iloc[0] > profile_counts.iloc[1])
    )

    rows_to_review = employee_rows.copy()
    rows_to_review.insert(
        2,
        "Campos_incoherentes",
        ", ".join(varying_fields),
    )

    if has_clear_majority:
        majority_profile = profile_counts.index[0]
        outlier_mask = profile_hash.ne(majority_profile)
        candidate_indices.extend(employee_rows.index[outlier_mask].tolist())
        rows_to_review.insert(
            3,
            "Clasificacion_revision",
            outlier_mask.map({True: "CANDIDATO_A_ELIMINAR", False: "PERFIL_MAYORITARIO"}).values,
        )
    else:
        ambiguous_employee_ids.append(employee_id)
        rows_to_review.insert(3, "Clasificacion_revision", "CASO_AMBIGUO")

    review_frames.append(rows_to_review)

review_columns = [
    "ID_Employee", "ID_Absence", "Campos_incoherentes",
    "Clasificacion_revision", *personal_columns,
]

if review_frames:
    registros_para_revision = (
        pd.concat(review_frames)
        .loc[:, review_columns]
        .sort_values(["ID_Employee", "ID_Absence"])
    )
else:
    registros_para_revision = pd.DataFrame(columns=review_columns)

candidatos_eliminacion = (
    df_RRHH.loc[candidate_indices]
    .sort_values(["ID_Employee", "ID_Absence"])
    .copy()
)
casos_ambiguos = registros_para_revision.loc[
    registros_para_revision["Clasificacion_revision"].eq("CASO_AMBIGUO")
].copy()

print(f"Empleados con alguna incoherencia: "
      f"{registros_para_revision['ID_Employee'].nunique()}")
print(f"Registros propuestos para eliminación: {len(candidatos_eliminacion)}")
print(f"Empleados con casos ambiguos: {len(set(ambiguous_employee_ids))}")

print("Todos los registros de los empleados afectados:")
display(registros_para_revision)

print("Candidatos propuestos para eliminación (requieren confirmación):")
display(candidatos_eliminacion)

if not casos_ambiguos.empty:
    print("Casos ambiguos: revisar manualmente; no se eliminarán automáticamente.")
    display(casos_ambiguos)


Empleados con alguna incoherencia: 1
Registros propuestos para eliminación: 1
Empleados con casos ambiguos: 0
Todos los registros de los empleados afectados:


,ID_Employee,ID_Absence,Campos_incoherentes,Clasificacion_revision,Transportation_Expense,Distance_Residence_Work,Service_Time,Age,Education,Son,Social_Drinker,Social_Smoker,Pet,Weight,Height,Body_Mass_Index
251,29,251,"Distance_Residence_Work, Service_Time, Age, Ed...",PERFIL_MAYORITARIO,225.0,15,15,41,4,2,True,False,2,94,182,28
252,29,252,"Distance_Residence_Work, Service_Time, Age, Ed...",PERFIL_MAYORITARIO,225.0,15,15,41,4,2,True,False,2,94,182,28
430,29,430,"Distance_Residence_Work, Service_Time, Age, Ed...",PERFIL_MAYORITARIO,225.0,15,15,41,4,2,True,False,2,94,182,28
536,29,536,"Distance_Residence_Work, Service_Time, Age, Ed...",PERFIL_MAYORITARIO,225.0,15,15,41,4,2,True,False,2,94,182,28
664,29,664,"Distance_Residence_Work, Service_Time, Age, Ed...",CANDIDATO_A_ELIMINAR,225.0,26,9,28,1,1,False,False,2,69,169,24


Candidatos propuestos para eliminación (requieren confirmación):


,ID_Absence,ID_Employee,Reason_Absence,Reason_Absence_Desc,Month_Absence,Day_Week,Day_Week_Desc,Seasons,Season_Desc,Transportation_Expense,...,Education,Education_Desc,Son,Social_Drinker,Social_Smoker,Pet,Weight,Height,Body_Mass_Index,Absenteeism_Hours
664,664,29,0,Sin ausencia registrada,9,2,Lunes,4,Otoño,225.0,...,1,Educación secundaria,1,False,False,2,69,169,24,0


### Confirmación de la eliminación

Ejecuta primero la detección anterior y revisa **registros_para_revision** y **candidatos_eliminacion**. La siguiente celda parte de los candidatos detectados, pero no modifica el DataFrame mientras **CONFIRMAR_ELIMINACION = False**.

Puedes retirar manualmente identificadores de **ID_ABSENCE_A_ELIMINAR** antes de confirmar. La celda impide eliminar cualquier registro que no haya sido propuesto por la detección.


In [15]:
# Por defecto se seleccionan todos los candidatos con perfil minoritario.
# Elimina manualmente de esta lista cualquier ID_Absence que quieras conservar.
ID_ABSENCE_A_ELIMINAR = candidatos_eliminacion["ID_Absence"].tolist()

# Cambiar a True únicamente después de revisar las tablas mostradas arriba.
CONFIRMAR_ELIMINACION = True

if not CONFIRMAR_ELIMINACION:
    print(
        "Eliminación pendiente de confirmación. "
        f"Candidatos seleccionados: {ID_ABSENCE_A_ELIMINAR}"
    )
else:
    candidate_ids = set(candidatos_eliminacion["ID_Absence"].tolist())
    selected_ids = set(ID_ABSENCE_A_ELIMINAR)
    non_candidate_ids = sorted(selected_ids - candidate_ids)
    if non_candidate_ids:
        raise ValueError(
            "Se intentan eliminar registros no propuestos por la validación: "
            f"{non_candidate_ids}"
        )
    if not selected_ids:
        print("No se ha seleccionado ningún registro para eliminar.")
    else:
        registros_incoherentes_eliminados = df_RRHH.loc[
            df_RRHH["ID_Absence"].isin(selected_ids)
        ].copy()

        if len(registros_incoherentes_eliminados) != len(selected_ids):
            found_ids = set(registros_incoherentes_eliminados["ID_Absence"])
            missing_ids = sorted(selected_ids - found_ids)
            raise ValueError(f"No existen estos ID_Absence en df_RRHH: {missing_ids}")

        df_RRHH = (
            df_RRHH.loc[~df_RRHH["ID_Absence"].isin(selected_ids)]
            .reset_index(drop=True)
            .copy()
        )

        assert not df_RRHH["ID_Absence"].isin(selected_ids).any()
        print(f"Registros eliminados tras confirmación: {len(selected_ids)}")
        display(registros_incoherentes_eliminados)


Registros eliminados tras confirmación: 1


,ID_Absence,ID_Employee,Reason_Absence,Reason_Absence_Desc,Month_Absence,Day_Week,Day_Week_Desc,Seasons,Season_Desc,Transportation_Expense,...,Education,Education_Desc,Son,Social_Drinker,Social_Smoker,Pet,Weight,Height,Body_Mass_Index,Absenteeism_Hours
664,664,29,0,Sin ausencia registrada,9,2,Lunes,4,Otoño,225.0,...,1,Educación secundaria,1,False,False,2,69,169,24,0


## Validación final adicional recomendada

Esta celda no sustituye a otra: concentra invariantes mínimas antes de exportar o escribir en MySQL. Debe colocarse y ejecutarse justo antes de las salidas. Si alguna condición falla, detiene el proceso con un mensaje explícito.


In [16]:
required_columns = [
   'ID_Absence', 'ID_Employee', 'Reason_Absence', 'Reason_Absence_Desc',
       'Month_Absence', 'Day_Week', 'Day_Week_Desc', 'Seasons', 'Season_Desc',
       'Transportation_Expense', 'Distance_Residence_Work',
       'Service_Time', 'Age', 'Work_load_Average_Day', 'Hit_Target',
       'Disciplinary_Failure', 'Education', 'Education_Desc', 'Son',
       'Social_Drinker', 'Social_Smoker', 'Pet', 'Weight', 'Height',
       'Body_Mass_Index', 'Absenteeism_Hours']

missing_columns = sorted(set(required_columns) - set(df_RRHH.columns))
if missing_columns:
    raise KeyError(f"Faltan columnas obligatorias: {missing_columns}")

assert df_RRHH.columns.is_unique, "Hay nombres de columna duplicados"
assert df_RRHH["ID_Absence"].is_unique, "ID_Absence no es único"
assert df_RRHH["ID_Employee"].notna().all(), "Hay empleados sin ID"
assert df_RRHH["Month_Absence"].between(0, 12).all(), "Mes fuera de 0..12"
assert df_RRHH["Day_Week"].between(2, 6).all(), "Día fuera de 2..6"
assert df_RRHH["Absenteeism_Hours"].ge(0).all(), "Horas negativas"
assert not df_RRHH.duplicated().any(), "Persisten duplicados exactos"

summary = pd.DataFrame({
    "filas": [len(df_RRHH)],
    "columnas": [df_RRHH.shape[1]],
    "empleados": [df_RRHH["ID_Employee"].nunique()],
    "nulos_totales": [int(df_RRHH.isna().sum().sum())],
})
display(summary)


,filas,columnas,empleados,nulos_totales
0,1055,26,386,0


## Guardado en Base de Datos

In [17]:
TARGET_TABLE = "RRHH_CLEAN_13072026"

write_url = URL.create(
    drivername="mysql+pymysql",
    username=os.environ["MYSQL_USER"],
    password=os.environ["MYSQL_PASSWORD"],
    host=os.environ["MYSQL_HOST"],
    port=int(os.environ["MYSQL_PORT"]),
    database=os.environ["MYSQL_DATABASE"],
    query={"charset": "utf8mb4"},
)

write_engine = create_engine(write_url, pool_pre_ping=True)

if CONFIRMAR_ELIMINACION:
    with write_engine.begin() as connection:
        df_RRHH.to_sql(
            name=TARGET_TABLE,
            con=connection,
            if_exists="fail",  # Evita duplicar la carga por una reejecución accidental.
            index=False,
            chunksize=1000,
            method="multi",
        )

    print(f"{len(df_RRHH)} filas insertadas en {TARGET_TABLE}")
else:
    print(
        "No se ha confirmado la eliminación de registros incoherentes. "
        "No se realizará la carga en la base de datos."
    )


1055 filas insertadas en RRHH_CLEAN_13072026


## Guardado en CSV

In [ ]:
ACTIVATE_CSV_EXPORT = True  # Poner a True para exportar el CSV final.

OUTPUT_DIR = Path("../Data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
csv_path = OUTPUT_DIR / "DatasetClean_2026-07-13.csv"

if ACTIVATE_CSV_EXPORT:
    df_RRHH.to_csv(
        csv_path,
        index=False,
        encoding="utf-8-sig",
    )
    if CONFIRMAR_ELIMINACION:
        print("Se han eliminado los registros incoherentes y se ha generado el CSV final.")
    else:
        print("Se ha generado el CSV final sin eliminar registros incoherentes.")

    print(f"CSV guardado en {csv_path.resolve()} ({len(df_RRHH)} filas)")
else:
    print(
        "No se generará el CSV."
    )


Se han eliminado los registros incoherentes y se ha generado el CSV final.
CSV guardado en C:\Users\raulm\Desktop\Simulator\ProjecteData\Equip_33\Data\DataClean_2026-07-13.csv (1055 filas)
